In [1]:
import collections
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages

In [35]:
VCF_PATH = "../../../../hybrid_diversity/5_snp_analyses/3_aims/3_vcf/hybrid_variants_AIMs.vcf"
OUT_PDF  = "loh_figure_heatmap_allstrains.pdf"
 
WIN_SIZE  = 200000   # window size in bp
STEP_SIZE = 200000   # step size in bp  (non-overlapping)
MIN_SNPS  = 25        # minimum SNPs in window to call a state
 
# Parse from VCF header contigs
CONTIG_LENGTHS = {}
with open(VCF_PATH) as fh:
    for line in fh:
        if line.startswith("##contig"):
            # e.g. ##contig=<ID=CM083864.1,length=3636776>
            import re
            m = re.search(r'ID=([^,>]+).*length=(\d+)', line)
            if m:
                CONTIG_LENGTHS[m.group(1)] = int(m.group(2))
        elif line.startswith("#CHROM"):
            break
            
            
# Colour scheme
COL = {
    "het":     "#DDAA33",
    "hom_ref": "#BB5566",
    "hom_alt": "#004488",
    "miss":    "#000000",
}


In [36]:
# Figure aesthetics
TRACK_H    = 0.28   # inches per sample track
LABEL_W    = 0.9    # inches for left-side sample labels
CHR_GAP    = 0.05   # inches between chromosome panels
TOP_PAD    = 0.65   # inches at top for chr labels
BOT_PAD    = 0.50   # inches at bottom for axis + legend
LEFT_PAD   = 0.15
RIGHT_PAD  = 0.15
FONT_SIZE  = 7
CHR_LABEL_FS = 7.5

In [37]:
def gt_class(gt: str) -> str:
    """Return 'hom_ref', 'hom_alt', 'het', or 'miss'."""
    if not gt or gt in ("./.", ".|.", "."):
        return "miss"
    alleles = [a for a in gt.replace("|", "/").split("/") if a != "."]
    if len(alleles) < 2:
        return "miss"
    uniq = set(alleles)
    if len(uniq) == 1:
        return "hom_ref" if alleles[0] == "0" else "hom_alt"
    return "het"


In [38]:
# Define corner colours as numpy arrays
def hex_to_rgb(h):
    return np.array([int(h[i:i+2], 16) for i in (1, 3, 5)]) / 255

def lighten(hex_col, amount):
    rgb = hex_to_rgb(hex_col)
    return rgb + (1 - rgb) * amount

C_HET     = lighten("#DDAA33", 0.2)
C_HOM_REF = lighten("#BB5566", 0.0)
C_HOM_ALT = lighten("#004488", 0.0)
C_MISS    = np.array([0.0, 0.0, 0.0])

def window_color(counts, total):
    called = counts["het"] + counts["hom_ref"] + counts["hom_alt"]
    if called < MIN_SNPS:
        return C_MISS
    f_het = counts["het"]     / called
    f_ref = counts["hom_ref"] / called
    f_alt = counts["hom_alt"] / called
    rgb = f_het * C_HET + f_ref * C_HOM_REF + f_alt * C_HOM_ALT
    return np.clip(rgb, 0, 1)


In [39]:
print(f"Parsing {VCF_PATH} ...")

SAMPLE_ORDER = ['12B1', 'PK2298B4', 'CCMP3037', 'CC2192B4', 'CC2192B5', 'CC2192C1', 'CC2192D2', 'CC2192D3', '8020', 'ARC66', 'ARC81', 'ARC82', 'ARC83', 'ARC84', 'PK2298C6', 'PK2300D3', 'UTEX2797', 'UTEX2797-SC1', 'UTEX2797-SC2', 'UTEX2797-SC3', 'UTEX2797-SC4', '12A1', '12A1-SC1', '12A1-SC2', '12A1-SC3', '13A5', '8027', 'PK2297D4', 'PK2299B3', 'UTEX2827', 'RCC3703', 'CCMP2941', 'NIES1812']

samples = []
chroms_order = []
by_chrom: dict[str, list] = collections.defaultdict(list)

with open(VCF_PATH) as fh:
    for line in fh:
        if line.startswith("##"):
            continue
        if line.startswith("#CHROM"):
            samples = line.strip().split("\t")[9:]
            continue
        cols = line.strip().split("\t")
        chrom, pos = cols[0], int(cols[1])
        fmt    = cols[8].split(":")
        gt_idx = fmt.index("GT")
        gts    = [c.split(":")[gt_idx] if gt_idx < len(c.split(":")) else "./."
                  for c in cols[9:]]
        if chrom not in chroms_order:
            chroms_order.append(chrom)
        by_chrom[chrom].append((pos, gts))

# Reorder AFTER parsing — samples is now populated
sample_index = {s: i for i, s in enumerate(samples)}
samples = [s for s in SAMPLE_ORDER if s in sample_index]

# Also reindex by_chrom gts to match new sample order
new_indices = [sample_index[s] for s in samples]
for chrom in chroms_order:
    by_chrom[chrom] = [(pos, [gts[i] for i in new_indices]) for pos, gts in by_chrom[chrom]]

n_samples = len(samples)
n_chroms  = len(chroms_order)
print(f"  {len(samples)} samples, {n_chroms} chromosomes, "
      f"{sum(len(v) for v in by_chrom.values()):,} variants")

Parsing ../../../../hybrid_diversity/5_snp_analyses/3_aims/3_vcf/hybrid_variants_AIMs.vcf ...
  33 samples, 34 chromosomes, 277,926 variants


In [40]:
# calculate sliding windows
 
# windows[chrom][sample_idx] = list of (x_start_frac, x_end_frac, color)
windows: dict[str, list[list]] = {}
 
for chrom in chroms_order:
    vlist   = by_chrom[chrom]
    chr_len = CONTIG_LENGTHS[chrom]
    sample_wins = [[] for _ in samples]
 
    for start in range(1, chr_len - WIN_SIZE + 2, STEP_SIZE):
        end = start + WIN_SIZE - 1
        in_win = [(pos, gts) for pos, gts in vlist if start <= pos <= end]
        total  = len(in_win)
        #print(total)
 
        x0 = (start - 1) / chr_len
        x1 = min(end, chr_len) / chr_len
 
        for si in range(n_samples):
            if total == 0:
                color = COL["miss"]
            else:
                counts = {"het": 0, "hom_ref": 0, "hom_alt": 0, "miss": 0}
                for _, gts in in_win:
                    counts[gt_class(gts[si])] += 1
                color = window_color(counts, total)
            sample_wins[si].append((x0, x1, color))
 
    windows[chrom] = sample_wins
 

In [41]:
# ── layout calculations ────────────────────────────────────────────────────────
 
total_bp   = sum(CONTIG_LENGTHS[c] for c in chroms_order)
total_gaps = CHR_GAP * (n_chroms - 1)
 
# Available width for chromosome tracks (inches)
PLOT_W = 32   # total figure width
avail_w = PLOT_W - LABEL_W - LEFT_PAD - RIGHT_PAD - total_gaps
 
# Width of each chromosome panel (inches), proportional to length
chr_widths = {c: avail_w * CONTIG_LENGTHS[c] / total_bp for c in chroms_order}
 
# Height
track_block = TRACK_H * n_samples
fig_h = TOP_PAD + track_block + BOT_PAD
 
fig_w = PLOT_W

In [42]:
# ── draw figure ───────────────────────────────────────────────────────────────
 
print("Rendering figure ...")
 
fig = plt.figure(figsize=(fig_w, fig_h))
fig.patch.set_facecolor("white")
 
# Compute x positions of each chromosome panel in figure-fraction coords
x_cursor = (LEFT_PAD + LABEL_W) / fig_w
chr_x0: dict[str, float] = {}
for chrom in chroms_order:
    chr_x0[chrom] = x_cursor
    x_cursor += chr_widths[chrom] / fig_w + CHR_GAP / fig_w

# For each chromosome, draw each sample as a row of coloured rectangles
for chrom in chroms_order:
    cw_frac = chr_widths[chrom] / fig_w
    cx0     = chr_x0[chrom]
 
    for si, sname in enumerate(samples):
        # bottom of this track in figure coords (top sample first)
        y_bottom_in = BOT_PAD + (n_samples - 1 - si) * TRACK_H
        y0_frac = y_bottom_in / fig_h
        h_frac  = TRACK_H / fig_h * 0.88  # slight gap between tracks
 
        for (xf0, xf1, color) in windows[chrom][si]:
            rect_x = cx0 + xf0 * cw_frac
            rect_w = (xf1 - xf0) * cw_frac
            rect = mpatches.FancyArrowPatch   # not used; use Rectangle
            fig.add_artist(
                mpatches.Rectangle(
                    (rect_x, y0_frac),
                    rect_w, h_frac,
                    transform=fig.transFigure,
                    facecolor=color,
                    edgecolor="none",
                    linewidth=0,
                    zorder=2,
                )
            )
 
    # Chromosome label above the panel
    mid_x = cx0 + cw_frac / 2
    label_y = (BOT_PAD + n_samples * TRACK_H + 0.05) / fig_h
    # Short chr name: strip 'CM083' prefix for compactness
    short = chrom.replace("CM083", "").replace(".1", "")
    length_mb = CONTIG_LENGTHS[chrom] / 1e6
    fig.text(
        mid_x, label_y,
        f"Chr {short}\n{length_mb:.2f} Mb",
        ha="center", va="bottom",
        fontsize=CHR_LABEL_FS, color="#333333",
        transform=fig.transFigure,
        linespacing=1.3,
    )
 
    # Mb axis ticks below bottom sample
    ax_y = (BOT_PAD - 0.08) / fig_h
    tick_vals_mb = np.arange(0, CONTIG_LENGTHS[chrom] / 1e6 + 0.1, 1.0)
    for mb in tick_vals_mb:
        frac = mb * 1e6 / CONTIG_LENGTHS[chrom]
        if frac > 1.0:
            break
        tx = cx0 + frac * cw_frac
        # tick line
        fig.add_artist(
            plt.Line2D([tx, tx],
                       [(BOT_PAD - 0.06) / fig_h, (BOT_PAD - 0.02) / fig_h],
                       transform=fig.transFigure,
                       color="#888888", linewidth=0.5, zorder=3)
        )
        if mb == int(mb):
            fig.text(tx, (BOT_PAD - 0.14) / fig_h,
                     f"{int(mb)}",
                     ha="center", va="top",
                     fontsize=FONT_SIZE - 1, color="#666666",
                     transform=fig.transFigure)
 
    # Thin border around whole chromosome block
    block_x = cx0 - 0.001
    block_y = BOT_PAD / fig_h
    block_w = cw_frac + 0.002
    block_h = (n_samples * TRACK_H) / fig_h
    fig.add_artist(
        mpatches.Rectangle(
            (block_x, block_y), block_w, block_h,
            transform=fig.transFigure,
            facecolor="none",
            edgecolor="#cccccc",
            linewidth=0.4,
            zorder=4,
        )
    )

# Sample labels (left side)
first_cx = chr_x0[chroms_order[0]]
for si, sname in enumerate(samples):
    y_center = (BOT_PAD + (n_samples - 1 - si) * TRACK_H + TRACK_H * 0.44) / fig_h
    fig.text(
        first_cx - 0.012,
        y_center,
        sname,
        ha="right", va="center",
        fontsize=FONT_SIZE, color="#333333",
        transform=fig.transFigure,
        fontfamily="monospace",
    )

# Horizontal gridlines separating samples (light)
for si in range(n_samples + 1):
    gy = (BOT_PAD + si * TRACK_H) / fig_h
    fig.add_artist(
        plt.Line2D(
            [chr_x0[chroms_order[0]] - 0.001,
             chr_x0[chroms_order[-1]] + chr_widths[chroms_order[-1]] / fig_w + 0.001],
            [gy, gy],
            transform=fig.transFigure,
            color="#dddddd", linewidth=0.3, zorder=5,
        )
    )

# Legend
legend_y = 0.025
legend_items = [
    ("Heterozygous",      COL["het"]),
    ("Hom. reference",    COL["hom_ref"]),
    ("Hom. alternate",    COL["hom_alt"]),
    ("No data / < " + str(MIN_SNPS) + " SNPs", COL["miss"]),
]
lx = (LEFT_PAD + LABEL_W) / fig_w
for label, color in legend_items:
    fig.add_artist(
        mpatches.Rectangle(
            (lx, legend_y / fig_h * fig_h),   # recompute cleanly
            0.018, 0.032 / fig_h,
            transform=fig.transFigure,
            facecolor=color, edgecolor="#999999", linewidth=0.4,
        )
    )
    # Actually use absolute figure coords:
    fig.add_artist(
        mpatches.Rectangle(
            (lx, 0.022),
            0.018, 0.016,
            transform=fig.transFigure,
            facecolor=color, edgecolor="#aaaaaa", linewidth=0.4, zorder=6,
        )
    )
    fig.text(lx + 0.022, 0.030, label,
             ha="left", va="center",
             fontsize=FONT_SIZE, color="#333333",
             transform=fig.transFigure)
    lx += 0.13

# Axis label
fig.text(
    (LEFT_PAD + LABEL_W + avail_w / 2) / fig_w + LEFT_PAD / fig_w * 0,
    0.005,
    "Genomic position (Mb)",
    ha="center", va="bottom",
    fontsize=FONT_SIZE, color="#555555",
    transform=fig.transFigure,
)
 
# Title / caption
fig.text(
    0.5, (BOT_PAD + n_samples * TRACK_H + TOP_PAD * 0.72) / fig_h,
    f"Loss of heterozygosity — sliding window ({WIN_SIZE//1000} kb)",
    ha="center", va="top",
    fontsize=8.5, fontweight="bold", color="#222222",
    transform=fig.transFigure,
)
 

Rendering figure ...


Text(0.5, 0.9824831568816169, 'Loss of heterozygosity — sliding window (200 kb)')

In [43]:
# ── save PDF ──────────────────────────────────────────────────────────────────
 
with PdfPages(OUT_PDF) as pdf:
    pdf.savefig(fig, dpi=300, bbox_inches="tight",
                facecolor="white", edgecolor="none")
    d = pdf.infodict()
    d["Title"]   = "LOH sliding window figure"
    d["Subject"] = "Genotype heatmap across genome"

plt.close(fig)
print(f"Saved: {OUT_PDF}")


Saved: loh_figure_heatmap_allstrains.pdf
